In [2]:
from langchain_community.document_loaders import TextLoader

C:\Users\admin\AppData\Local\Temp\ipykernel_10728\2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
from pathlib import Path


def get_file_from_project(file_name, project_root=None):
    
    project_root = Path(project_root) if project_root else Path.cwd().resolve().parents[1]

    matches = list(project_root.glob(f"**/{file_name}"))
    if not matches:
        raise FileNotFoundError(f"'{file_name}' not found in {project_root}")

    return matches[0]


In [4]:
target_file = get_file_from_project("sample_dataset.txt")
print(target_file)

D:\2027\Projects\RAG\data\sample_dataset.txt


In [5]:
## Loading a single text file
loader=TextLoader(get_file_from_project("sample_dataset.txt"), encoding="utf-8")

In [ ]:
documents = loader.load()  # Returns a list of LangChain Document objects

### Sample Document Structure

```python
Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "Tej Dave",
        "date_created": "2024-01-01",
        "custom_field": "any_value"
    }
)
```

In [11]:
print(f"📄 Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

📄 Loaded 1 document
Content preview: Title: Bhagavad Gita Knowledge Dataset

Overview:
The Bhagavad Gita, commonly known as the Gita, is ...
Metadata: {'source': 'D:\\2027\\Projects\\RAG\\data\\sample_dataset.txt'}


In [26]:
# Three text splitting methods (production-style setup)
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

# Required parameters used across all methods
CHUNK_SIZE = 1000  # Target maximum size for each chunk (characters for char splitters).
CHUNK_OVERLAP = 100  # Shared context carried from one chunk to the next.
PREVIEW_CHARS = 120  # Number of characters shown when previewing first chunk.

if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be > 0")
if CHUNK_OVERLAP < 0 or CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError("CHUNK_OVERLAP must be >= 0 and < CHUNK_SIZE")

sample_text = documents[0].page_content if documents else "This is a sample text for splitting."
if not isinstance(sample_text, str) or not sample_text.strip():
    raise ValueError("Input text must be a non-empty string")



In [32]:

def preview_chunks(method_name, chunks, preview_chars=PREVIEW_CHARS):
    print(method_name)
    print("Chunks:", len(chunks))
    if chunks:
        print("First chunk:", chunks[0][:preview_chars])
    else:
        print("First chunk: <empty>")

In [33]:
# 1) CharacterTextSplitter
# Best for fixed-size character chunks with explicit separator handling.
char_splitter = CharacterTextSplitter(
    separator="",  # Primary boundary to split on (newline).
    is_separator_regex=False,  # Treat separator as plain text, not regex.
    chunk_size=CHUNK_SIZE,  # Max characters allowed in each chunk.
    chunk_overlap=CHUNK_OVERLAP,  # Characters repeated between adjacent chunks.
    keep_separator=False,  # Do not keep separator token inside chunk output.
    strip_whitespace=True,  # Trim leading/trailing spaces from each chunk.
)
char_chunks = char_splitter.split_text(sample_text)
preview_chunks("CharacterTextSplitter", char_chunks)

CharacterTextSplitter
Chunks: 9
First chunk: Title: Bhagavad Gita Knowledge Dataset

Overview:
The Bhagavad Gita, commonly known as the Gita, is a 700-verse Sanskrit


In [34]:
# 2) RecursiveCharacterTextSplitter
# Best for preserving readable boundaries (paragraphs/sentences/words).
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],  # Fallback order: paragraph -> line -> word -> char.
    keep_separator=True,  # Keep delimiters to preserve natural text structure.
    is_separator_regex=False,  # Treat separators as literal strings.
    chunk_size=CHUNK_SIZE,  # Max characters allowed in each chunk.
    chunk_overlap=CHUNK_OVERLAP,  # Shared context across chunk boundaries.
    strip_whitespace=True,  # Remove extra edge whitespace per chunk.
)
recursive_chunks = recursive_splitter.split_text(sample_text)
preview_chunks("RecursiveCharacterTextSplitter", recursive_chunks)

RecursiveCharacterTextSplitter
Chunks: 9
First chunk: Title: Bhagavad Gita Knowledge Dataset

Overview:
The Bhagavad Gita, commonly known as the Gita, is a 700-verse Sanskrit


In [35]:
# 3) TokenTextSplitter
# Best when you need token-aware chunking for LLM context limits.
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",  # Tokenizer used to count/segment tokens.
    chunk_size=40,  # Max tokens per chunk.
    chunk_overlap=5,  # Overlapping tokens between neighboring chunks.
)
token_chunks = token_splitter.split_text(sample_text)
preview_chunks("TokenTextSplitter", token_chunks)

TokenTextSplitter
Chunks: 54
First chunk: Title: Bhagavad Gita Knowledge Dataset

Overview:
The Bhagavad Gita, commonly known as the Gita, is a 700-verse Sanskrit


In [36]:
for chun_no,data in enumerate(char_chunks[10:13]):
    print(f"Chunk {chun_no+1}: {data}")

In [25]:
for chun_no,data in enumerate(recursive_chunks[10:15]):
    print(f"Chunk {chun_no+1}: {data}")

Chunk 1: Traditional Verse Count: 700 verses
Setting: Battlefield of Kurukshetra
Chunk 2: Author Tradition: Traditionally attributed to Sage Vyasa as part of the
Chunk 3: of the Mahabharata
Chunk 4: Composition Period: Scholars have different views; generally estimated between
Chunk 5: between the last centuries BCE and early centuries CE.


In [37]:
for chun_no,data in enumerate(token_chunks[10:13]):
    print(f"Chunk {chun_no+1}: {data}")

Chunk 1:  traditions.

3. Sanjaya:
- Advisor and charioteer of King Dhritarashtra.
- Receives divine vision from Sage Vyasa to witness and narrate the events of the
Chunk 2: ate the events of the Kurukshetra battlefield.

4. Dhritarashtra:
- King of Hastinapura.
- Father of the Kauravas.
- Blind from birth
Chunk 3: .
- Blind from birth and concerned about the outcome of the war.

5. Pandavas:
- Five brothers: Yudhishthira, Bhima, Arjuna, Nakula
